# 第12回: Planning and Reflection②

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session12/session12_planning_reflection_2.ipynb)

これまでのエージェントは、依頼を受けてから ReAct ループを回し、必要な Tool を呼び、答えを返してきた。この形は「聞かれたことに答える」までは強いが、「目的を達成する」には足りない。何を達成すれば終わりなのかが決まっておらず、途中の成果物が十分かを判断する仕組みも、うまく進んでいないときに軌道を変える仕組みもないためである。

今回は、エージェントを目的志向のシステムにする4つの能力を搭載する。

- **Goal Setting**: 何を達成すれば完了なのかを、観測できる条件として定める
- **Planning**: 目的へ至る手順を、実行可能なステップ列へ分解する
- **Reflection**: 出した成果物を自分で批評し、作り直す
- **Monitoring**: 実行結果を成功条件と照らし、続ける・作り直す・終える・人間へ戻すを決める

---

## 0. 環境準備

In [ ]:
# @markdown 実行環境フラグ: Google Colab で実行する場合は True にする
IS_COLAB = False # @param {type:"boolean"}

In [ ]:
if IS_COLAB:
    !git clone https://github.com/cosmac-dev/ai-agent-seminar.git
    %cd ai-agent-seminar/session11

%pip install -q -e "."

In [ ]:
# @title APIキーの設定
import getpass
import os
from pathlib import Path

# ローカル実行で .env がある場合はそこから読み込む
if Path('.env').exists():
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except ImportError:
        pass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY を入力する: ')

# 第6節で使う guarded_agent の web_search / fetch_url は、import 時に Tavily を初期化する
if not os.environ.get('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY'] = getpass.getpass('TAVILY_API_KEY を入力する: ')

print('OpenAI APIキー設定完了' if os.environ.get('OPENAI_API_KEY') else 'OpenAI APIキー未設定')
print('Tavily APIキー設定完了' if os.environ.get('TAVILY_API_KEY') else 'Tavily APIキー未設定')

---

## 1. 目的志向のエージェント

複雑な依頼は、1回の応答や1回の Tool 呼び出しでは終わらない。まず何を達成すれば終わりかを定め、そこから中間ステップへ分解し、各ステップの成果物が十分かを判断し、全体が目的へ近づいているかを確かめる必要がある。

これらを1つの大きなプロンプトへ詰め込むこともできるが、そうすると「今どのステップにいるか」「何が未達か」「なぜやり直したか」がすべて会話文のコンテキストから認知・理解・判断をする必要がある。今回の実装では、目的・成功条件・計画・観測・判断を**状態**として持つことで、実行中に扱える制御情報へ変える。

責務を分けると、それぞれの役割は次のようになる。

| 能力 | 役割 | 状態として持つもの |
|---|---|---|
| Goal Setting | 何を達成すれば完了か | 目的、成功条件、制約 |
| Planning | どの順で何をするか | ステップ列、現在の位置 |
| Reflection | この成果物で十分か | 批評、指摘の履歴、改稿回数 |
| Monitoring | 全体として次に何をすべきか | 観測、未達ギャップ、次の遷移 |

Reflection と Monitoring は似ているが、見ている対象が違う。Reflection は**成果物の質**を見て、同じステップをやり直すかどうかを決める。Monitoring は**進行**を見て、次のステップへ進むか、計画を組み直すか、終わるか、人間へ戻すかを決める。この2つを1つのノードに混ぜると、「文章が硬い」といった品質の指摘と「計画が現実と合っていない」という進行の問題が同じ判断へ流れ込み、どちらの理由で止まったのかが追えなくなる。

### 統合ループ

4つの能力は、一度きりの直線的な処理ではない。計画は仮説であり、実行結果によって更新される。

[![](https://mermaid.ink/img/pako:eNp1ks9rFDEUx_-VIScFi_RapCfFiwVpbzUe4szb2cGZZMxmVOgW3J1apsVDVWxB-xPF2kMXBC22W_xnspnZ_he-ZHenLcUc8pL38vm-5OUtEV8EQGY80ojFK7_JpPIezVPu4ZDwIoOWuvWEEp1_1_m57p7gPPi7c7F_TsnT297U1KwXChbjiYdovAVQKuLhvWfy7mz1pVd9XtF5vyw2zPpeuXMw6CPeN8VJ9WsF8VESizudNGYcdR6j4RMN3T3V-arOc51vmWJLd_6YYnV4-LWmLeRoeA0-0g_QZCoS3OHTV3nd_WB6e8ODdzVsGQdLaMTgK-TnR6uJAF693N2u1o4QLtdOh0efanjMWL4t4WXUgrYTvBllvg-panuJ4JESErPMjVaTV14rUOdH9fbQbBR1ojHmpHzBsb7ZtVRX4xJsQdquLDej0PJZzBTSzSxxxR6cnV1sftSdnim-lZvH_8mZpDFYKhAcbDOU22-q312z_9O8X7dtMGKcZv2TlJM7HgllFGBvKZkB7hKQCXMOskSlpShRTUiAoouSgMnnlNjAsmVTxheFSC5xKbKwebnN0gDfcj9ioWT2VIPFLeuHwF59btzVrruX_wGwmi1A?type=png)](https://mermaid.live/edit#pako:eNp1ks9rFDEUx_-VIScFi_RapCfFiwVpbzUe4szb2cGZZMxmVOgW3J1apsVDVWxB-xPF2kMXBC22W_xnspnZ_he-ZHenLcUc8pL38vm-5OUtEV8EQGY80ojFK7_JpPIezVPu4ZDwIoOWuvWEEp1_1_m57p7gPPi7c7F_TsnT297U1KwXChbjiYdovAVQKuLhvWfy7mz1pVd9XtF5vyw2zPpeuXMw6CPeN8VJ9WsF8VESizudNGYcdR6j4RMN3T3V-arOc51vmWJLd_6YYnV4-LWmLeRoeA0-0g_QZCoS3OHTV3nd_WB6e8ODdzVsGQdLaMTgK-TnR6uJAF693N2u1o4QLtdOh0efanjMWL4t4WXUgrYTvBllvg-panuJ4JESErPMjVaTV14rUOdH9fbQbBR1ojHmpHzBsb7ZtVRX4xJsQdquLDej0PJZzBTSzSxxxR6cnV1sftSdnim-lZvH_8mZpDFYKhAcbDOU22-q312z_9O8X7dtMGKcZv2TlJM7HgllFGBvKZkB7hKQCXMOskSlpShRTUiAoouSgMnnlNjAsmVTxheFSC5xKbKwebnN0gDfcj9ioWT2VIPFLeuHwF59btzVrruX_wGwmi1A)

Monitoring は単なるログ記録ではなく、観測した結果をもとに、次の判断をする**制御ノード**。

- **complete**: 成功条件を満たしたので終了する
- **continue**: まだ作業が残っているので次のステップへ進む
- **replan**: 計画が現実と合わないので組み直す
- **escalate**: 判断が曖昧、またはリスクが高いので人間へ戻す

このノートブックでは、第2節で Session 11 から引き継いだ Reflection の内側ループを確認し、第3節で Goal Setting、第4節で Planning、第5節で Monitoring を足して外側ループを閉じる。第6節では実行部を Human-in-the-Loop と Sandbox 付きのエージェントへ差し替える。

### 適用判断

動的な計画は万能ではない。柔軟性と予測可能性はトレードオフの関係にあり、手順が既に分かっている処理では、エージェントに計画させるより固定ワークフローの方が速く、安く、結果も安定する。請求書の定型チェック、決まったデータ変換、既定の承認フローなどがこれにあたる。

判断の軸は単純で、**how を実行中に発見する必要があるか**である。

| 判断基準 | 向く設計 |
|---|---|
| 手順が既知で、毎回同じ順序でよい | 固定ワークフロー |
| how を実行中に発見する必要がある | Planning + Monitoring |
| 成功条件を観測できる | Goal Setting + Monitoring |
| 出力の品質が速度やコストより重要 | Reflection |
| 失敗時の影響が大きい、判断が曖昧 | Human-in-the-Loop |
| 生成されたコマンドやコードを実行する | Sandbox |

Reflection にも同じトレードオフがある。改稿のたびに LLM 呼び出しが増えるため、コストとレイテンシは反復回数に比例して増え、履歴が伸びてコンテキストを圧迫する。品質が速度より重要な場合に限って使う。

---

## 2. Reflection

Reflection は、エージェントが自分の出力を評価し、その評価をもとに改善する設計パターン。単純な連鎖では出力がそのまま次の工程へ流れるが、Reflection は途中にフィードバックループを入れる。

処理は4段階になる。

1. **実行**: 最初の成果物を作る
2. **評価・批評**: 成果物を基準に照らして分析する
3. **改善**: 批評をもとに作り直す
4. **反復**: 満足のいく結果になるか、停止条件に達するまで繰り返す

重要なのは、評価する主体を生成する主体から**論理的に分離する**ことである。同じ役割のまま「見直して」と頼んでも、モデルは自分の出力を妥当だと見なしやすい。プロンプトを分け、別のペルソナ（例: 厳密なコードレビュアー、事実確認担当）を与えると、指摘は具体的になり、見落としも減る。この構成を Producer-Critic と呼ぶ。

[![](https://mermaid.ink/img/pako:eNplkc9KAzEQxl8l5KRg9V6kFz0qiEeNSNxN29DuZslmVWh72AXRYg-K_6FaFFGrqAdBaD34MHGzfQyTVFvQnGYmv2_yTaYGHeYSmAewWGVbThlzARaWkQ_0ETisTKwiKJNPmfRk8irjh_QoHrw9pp2-6p8iuDYJcrkCCDhzI4dwzS79hLMbfKaQHXfU3oGML2Syr-Fh11_YKl2Oi0LLNKau2lmzO8LsjWUcTgV1NDRnA9t5aEDGT9nOvUyaMj6T8Z1q9gbdkz_PDdWmUR1BTjZpSPJAtXbV4bkeRwfZbR_B-sjWPxV2HBKIOmCRMJ-RvrS0WTP6kKxSj5oJ0ritnm_U5fXXx7u16OHtdSoIx4IyP9QCkJseTYN8OAVgiVNXf73gEdGZR7iHbQHWkPWBoCgTjyBdQtDFvIKguWgYbYD9Fca8sZyzqFQep1HgYkHmKS5xbKgiroamTlwqGF_8WbpdfuMbKtnPrg?type=png)](https://mermaid.live/edit#pako:eNplkc9KAzEQxl8l5KRg9V6kFz0qiEeNSNxN29DuZslmVWh72AXRYg-K_6FaFFGrqAdBaD34MHGzfQyTVFvQnGYmv2_yTaYGHeYSmAewWGVbThlzARaWkQ_0ETisTKwiKJNPmfRk8irjh_QoHrw9pp2-6p8iuDYJcrkCCDhzI4dwzS79hLMbfKaQHXfU3oGML2Syr-Fh11_YKl2Oi0LLNKau2lmzO8LsjWUcTgV1NDRnA9t5aEDGT9nOvUyaMj6T8Z1q9gbdkz_PDdWmUR1BTjZpSPJAtXbV4bkeRwfZbR_B-sjWPxV2HBKIOmCRMJ-RvrS0WTP6kKxSj5oJ0ritnm_U5fXXx7u16OHtdSoIx4IyP9QCkJseTYN8OAVgiVNXf73gEdGZR7iHbQHWkPWBoCgTjyBdQtDFvIKguWgYbYD9Fca8sZyzqFQep1HgYkHmKS5xbKgiroamTlwqGF_8WbpdfuMbKtnPrg)

Reflection が効くのは、品質・正確さ・制約への適合が速度やコストより重要な場面である。

| 領域 | 生成するもの | 批評の観点 |
|---|---|---|
| 文章生成 | 記事、告知文、要約 | 流れ、トーン、明確さ、抜けている論点 |
| コード生成 | 関数、スクリプト | バグ、境界条件、テスト結果、可読性 |
| 情報統合 | 長文の要約 | 原文の要点との照合、事実の取りこぼし |
| 計画立案 | 手順、戦略 | 実行可能性、制約違反、依存関係の矛盾 |

### 2.1 Reflectionの実装

階乗を計算する Python 関数の実装

- Critic には**品質チェックリスト**を明示的に渡し、その各項目に照らして判定させる。チェックリストは第3節以降の「成功条件」に相当するもので、Reflection が何を基準に合否を出すのかを外から決めるための仕組みを提供する。
- Critic の戻り値は文章ではなく構造化出力にする。`verdict` が制御に使え、`issues` がそのまま次の改稿の入力になる。
- Critic には、未達項目のうち**最も重要なものを1件だけ**挙げさせる。一度に全部直させるより、どの指摘が成果物のどこを変えたかを追いやすく、レビューの実務にも近い。

In [ ]:
# @title Reflection ループの実装
from typing import Literal

from IPython.display import Image, Markdown, display
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.runnables.graph_mermaid import CurveStyle
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

model_id = 'gpt-5.4-nano' # @param ['gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-5.6-luna', 'gpt-5.5', 'gpt-5.4', 'gpt-5.4-mini', 'gpt-5.4-nano']
llm = ChatOpenAI(model=model_id, temperature=0.2)

TASK_PROMPT = """`calculate_factorial` という名前の Python 関数を書いてください。
整数 `n` を受け取り、その階乗 (n!) を返します。
"""

# 批評の判定基準。第3節以降の「成功条件」にあたる
QUALITY_CHECKLIST = [
    'docstring に引数、戻り値、送出する例外が書かれている',
    '0 の階乗が 1 になる',
    '負の数には ValueError を送出する',
    'bool は int のサブクラスなので、True / False を拒否する',
    '再帰ではなく反復で実装し、大きな n でも再帰上限に達しない',
    '使用例が doctest 形式で書かれている',
]

PRODUCER_SYSTEM = 'あなたは Python の実装者です。要求を満たすコードだけを返します。説明文は書きません。'

CRITIC_SYSTEM = """あなたは Python に詳しいシニアソフトウェアエンジニアです。
提示されたコードを品質チェックリストの各項目に照らしてレビューしてください。
未達の項目があれば、そのうち最も重要なものを1件だけ issues に入れ、verdict を revise にします。
すべて満たしていれば verdict を accept とし、issues は空にします。"""


class Critique(BaseModel):
    """批評の結果。verdict がループの制御に、issues が次の改稿の入力になる。"""

    verdict: Literal['accept', 'revise'] = Field(
        description='全項目を満たすなら accept、そうでなければ revise。'
    )
    issues: list[str] = Field(description='最も重要な未達項目を1件だけ。accept のときは空。')
    guidance: str = Field(description='次の改稿で何をすべきかの指示。1文。')


# Critic だけ構造化出力にする。Producer の出力は成果物そのものなので素の文字列でよい
critic = llm.with_structured_output(Critique)


def checklist_text() -> str:
    return '\n'.join(f'- {item}' for item in QUALITY_CHECKLIST)


# ここから反復。何回目の改稿か、これまでどんな指摘を受けたか、いつ止めるか。
# これらを関数の引数で持ち回るとノードの責務がすぐ曖昧になるため、グラフの状態として持つ
class ReflectionState(TypedDict, total=False):
    task: str
    draft: str
    critique: Critique
    revision_notes: list[str]  # これまでに受けたすべての指摘（記憶）
    issue_log: list[list[str]]  # サイクルごとの指摘。比較のために残す
    iteration: int
    max_iterations: int
    use_memory: bool


def generate(state: ReflectionState) -> dict:
    """初回は生成、2回目以降は指摘を反映して書き直す。"""
    messages = [SystemMessage(content=PRODUCER_SYSTEM), HumanMessage(content=state['task'])]

    if state.get('draft'):
        if state.get('use_memory', True):
            # 記憶あり: 前回の成果物と、これまでの指摘をすべて渡す
            context = f'前回の成果物:\n{state["draft"]}\n\n'
            context += 'これまでに受けた指摘:\n' + '\n'.join(
                f'- {note}' for note in state.get('revision_notes', [])
            )
        else:
            # 記憶なし: 直近の指摘だけを渡す。前回の成果物も過去の指摘も見えない
            context = '次の指摘を踏まえて書いてください。\n' + '\n'.join(
                f'- {issue}' for issue in state['critique'].issues
            )
        messages.append(HumanMessage(content=context))

    response = llm.invoke(messages)
    return {'draft': str(response.content), 'iteration': state.get('iteration', 0) + 1}


def critique(state: ReflectionState) -> dict:
    """成果物をチェックリストに照らして批評し、指摘を記憶へ積む。"""
    result = critic.invoke([
        SystemMessage(content=CRITIC_SYSTEM),
        HumanMessage(content=(
            f'元の要求:\n{TASK_PROMPT}\n\n'
            f'品質チェックリスト:\n{checklist_text()}\n\n'
            f'レビュー対象:\n{state['draft']}'
        )),
    ])
    return {
        'critique': result,
        'revision_notes': state.get('revision_notes', []) + result.issues,
        'issue_log': state.get('issue_log', []) + [result.issues],
    }


def route_after_critique(state: ReflectionState) -> Literal['generate', 'finish']:
    """批評の verdict を分岐に使う。書き直すか、ここで終わるか。"""
    if state['critique'].verdict == 'accept':
        return 'finish'
    # 上限は付け足しの安全装置ではなく必須の構成要素。批評する側は「もっと良くできる点」を
    # いくらでも挙げられるので、accept が出るまで回す設計にすると止まらない（後の実行で確認する）
    if state.get('iteration', 0) >= state.get('max_iterations', 3):
        return 'finish'
    return 'generate'


reflection_builder = StateGraph(ReflectionState)
reflection_builder.add_node('generate', generate)
reflection_builder.add_node('critique', critique)

reflection_builder.add_edge(START, 'generate')
reflection_builder.add_edge('generate', 'critique')
reflection_builder.add_conditional_edges(
    'critique',
    route_after_critique,
    {'generate': 'generate', 'finish': END},
)

reflection_graph = reflection_builder.compile()

print('Model:', model_id)
display(Image(reflection_graph.get_graph().draw_mermaid_png(curve_style=CurveStyle.NATURAL)))

In [ ]:
# @title 反復ループを実行する
def run_reflection(*, use_memory: bool, max_iterations: int = 3) -> ReflectionState:
    return reflection_graph.invoke(
        {
            'task': TASK_PROMPT,
            'max_iterations': max_iterations,
            'use_memory': use_memory,
            'revision_notes': [],
            'issue_log': [],
        },
        config={'recursion_limit': 30},
    )


with_memory = run_reflection(use_memory=True)

print(f'改稿回数: {with_memory["iteration"]} / 最終判定: {with_memory["critique"].verdict}')
for index, issues in enumerate(with_memory['issue_log'], 1):
    print(f'\n--- {index} 回目の批評 ({len(issues)} 件) ---')
    for issue in issues:
        print('-', issue[:120])

display(Markdown(f'#### 最終成果物\n{with_memory["draft"]}'))

### 注意点

- **批評は尽きない**: 上の実行で見たとおり、Critic は要求を満たした後も「より厳密にできる点」を挙げ続ける。`accept` を待つ設計にすると止まらないため、`max_iterations` が必須になる
- **コストとレイテンシが反復に比例する**: 1サイクルごとに生成と批評で2回の LLM 呼び出しが増える。履歴も伸びるため、コンテキスト長の上限にも近づく
- **自己評価は万能ではない**: 同じモデルが書き手と評価者を兼ねると、見落としも共有される。静的チェックのように機械的に確かめられる観点は、LLM に判定させず自分で確かめる
- **止め方を決める**: 上限に達して終わった場合と、`accept` で終わった場合は意味が違う。どちらで終わったかを状態に残さないと、後段が品質を過信する

---

## 3. Goal Setting

Goal Setting は、ユーザーの依頼を「何を達成すれば完了か」という目標へ変換する能力。依頼文から、達成すべき最終状態である**目的**、完了判定に使う**成功条件**、守るべき**制約**を取り出す。

目標は、それを測る手段とセットで初めて意味を持つ。目的地を決めても、現在地が分からなければ到着したかを判断できない。後段の Monitoring が実行結果と成功条件を照合するため、成功条件は成果物や Tool の出力から観測できる粒度で定める。

目標の書き方としては SMART（具体的、測定可能、達成可能、関連性がある、期限や停止条件がある）が目安になる。ただし LLM エージェントではすべてを数値化できるとは限らない。重要なのは、評価する側が「完了」「未完了」「要修正」を判断できる程度に成功条件が明示されていることである。

ここで必要なのは、依頼をプロンプトへ渡し、目的・成功条件・制約の**構造化出力**を1回受け取る処理だけである。分岐もループもないため StateGraph は使わず、LangChain Expression Language（LCEL）のチェーンとして実装する。

In [ ]:
# @title Goal Setting の LCEL チェーン
from langchain_core.prompts import ChatPromptTemplate


class GoalSpec(BaseModel):
    """ユーザー依頼から取り出した目的と成功条件。"""

    objective: str = Field(description='達成すべき最終状態。1文で具体的に書く。')
    success_criteria: list[str] = Field(
        description='完了判定に使う、観測可能な成功条件。3から4件。'
    )
    constraints: list[str] = Field(description='守るべき制約や禁止事項。')


goal_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'ユーザー依頼を、達成すべき目的、観測可能な成功条件、制約へ分解する。\n'
        '成功条件は、成果物を読めば満たしているか判断できる粒度で書く。\n'
        '文字数や文の数のように、数えないと判定できない条件は成功条件に入れない。'
    )),
    ('human', '{request}'),
])

goal_chain = goal_prompt | llm.with_structured_output(GoalSpec)


def bullets(items) -> str:
    return '\n'.join(f'- {item}' for item in items) if items else '- なし'

In [ ]:
# @title Goal Setting を実行する
WRITING_REQUEST = (
    'AI エージェントに shell 実行を許すときの注意点を、社内共有用に3つの見出しでまとめてください。'
    '各見出しでは、どんなリスクがあるかと、それをどう抑えるかを書いてください。'
)

goal_result = goal_chain.invoke({'request': WRITING_REQUEST})

In [ ]:
# @title Goal Setting の結果

display(Markdown(
    f'**Objective**\n\n{goal_result.objective}\n\n'
    f'**Success Criteria**\n\n{bullets(goal_result.success_criteria)}\n\n'
    f'**Constraints**\n\n{bullets(goal_result.constraints)}'
))

### 実装の要点

- **成功条件は観測できる粒度で書く**: 「良いドキュメントを作る」は判定できない。「3つの見出しがある」なら、成果物を見れば判定できる。第5節の Monitoring はこの条件を使って合否を出すため、ここでの粒度が後段の精度を決める
- **判定者が確かめられる条件にする**: 「本文はちょうど100字」のような条件は、人間なら数えられるが LLM は正確に数えられない。judge が LLM である限り、こうした条件は永遠に未達と判定され続け、ループが止まらなくなる。字数を厳密に守らせたいなら、成功条件ではなくプログラムの検査として実装する
- **処理を LCEL で直結する**: `goal_prompt | llm.with_structured_output(GoalSpec)` が、入力の整形と構造化出力を1本の Runnable として表す。分岐や状態遷移は持たせない
- **結果を値として受け取る**: `goal_chain.invoke(...)` は `GoalSpec` を返す。Planning へ渡すまではグラフの状態へ変換する必要がない

### 注意点

成功条件の質が低いと、後段の計画と監視も安定しない。「安全に実行する」のような抽象的な条件は、何を観測すれば満たしたと判断できるかまで具体化する。

---

## 4. Planning

Planning は、高レベルの目標を実行可能なステップ列へ変換する能力。エージェントは第3節で定めた目的・成功条件・制約を受け取り、現在の状態から目標状態へ至る手順を作る。計画は事前に存在せず、依頼に応じて生成される。

### 計画は作り直せる

計画どおりに進まないとき、制約を計画へ追加し、残りの候補を評価し直して次の手順を組み直す。ただし第1節で触れたとおり、この柔軟性は予測可能性とトレードオフになる。実行のたびに手順が変わるため、同じ依頼でも同じ経路をたどるとは限らない。

### 計画を文章ではなく状態として持つ

「まず計画を立ててから作業して」とプロンプトへ書くだけでも、モデルは手順らしきものを出力する。ただしそれは応答本文の一部であり、後から参照するには本文を読み直して解釈する必要がある。

ここでは Goal Setting の結果を入力し、ステップ列の**構造化出力**を返す LCEL チェーンとして実装する。計画を実行して現在位置を管理する処理はまだ加えない。分岐と反復が必要になる第5節で、初めて LangGraph の状態へ格納する。

### 4.1 Planning の実装

`planning_chain` は、第3節で得た目的・成功条件・制約を受け取り、実行順のステップ列を作る。各ステップには中間成果物を1つだけ割り当てる。1ステップ目で成果物全体を作らせず、後段の Monitoring が実行結果を段階的に観測できる粒度にするためである。

計画は `WorkPlan` の構造化出力として受け取る。Goal Setting と同様に分岐もループもないため、`planning_prompt | llm.with_structured_output(WorkPlan)` という LCEL チェーンだけで表現できる。

In [ ]:
# @title Planning の LCEL チェーン
class WorkPlan(BaseModel):
    """目的を達成するための実行計画。"""

    steps: list[str] = Field(description='実行順のステップ。3件以内。各ステップは中間成果物を1つだけ作る。')


planning_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        '目的を達成するための実行計画を作る。'
        '各ステップは、1つの中間成果物だけを作る大きさにする。'
    )),
    ('human', (
        'Objective:\n{objective}\n\n'
        'Success Criteria:\n{success_criteria}\n\n'
        'Constraints:\n{constraints}'
    )),
])

planning_chain = planning_prompt | llm.with_structured_output(WorkPlan)


plan_result = planning_chain.invoke({
    'objective': goal_result.objective,
    'success_criteria': bullets(goal_result.success_criteria),
    'constraints': bullets(goal_result.constraints),
})


def numbered(items) -> str:
    return '\n'.join(f'{n}. {item}' for n, item in enumerate(items, 1)) if items else 'なし'


display(Markdown(
    f'**Plan**\n\n{numbered(plan_result.steps)}'
))

---

## 5. Monitoring

Monitoring は、実行結果・環境の状態・Tool の出力を観測し、第3節で定めた成功条件と照らして次の一手を決める仕組みである。第4節の計画は仮説であり、実行結果によって更新される。ここで初めて Goal Setting・Planning・Reflection・Monitoring の4つを状態遷移として統合する。

[![](https://mermaid.ink/img/pako:eNp1UktLw0AQ_ivLnhXvRTyIV2-eNFLWZNoGk2zYbFRsBdOKRPQgIooPlIKIivbgC6WgP2ZpUv-Fs2u0FTGXmZ2d7zHZqVObO0BLhFY8vmzXmJBkZtIKCH6wAvacRXWIJZQjCeH4ghibUM1X1dpUrZZqHarmXq9z3m_vWHSejI5OEAEVD2yJuCIb4LJ0Nzs7zbeuVdLpP9wgtH-5rZrbCP0SLBCapyFgyY2gYUz8vWW2DaFsEJ8HruQC1YqsHApeFRBFRvFj_R6taY_pc--trZKjYbkCYggt_A-BdIMYSiS7xc7OrymTF4sOe_kNFRB6LCiR_lWa73dRLn_aUMl7fvKIihqor-e--8qSRYv_mIDIZh6TaKKXXmQHd8iVpd2CpRb7hsbEctHq8uDfgfzQA82Vn3Ty442PZB9fQBNV3IB5SGSiuwo_BNqeecXBpEZsqEZHCK0K18GVkSIGPPkgfGYKtG4JjbKorIGPtCVMHSZwWn2xprEhC2Y59wdwweNqbXCMQwfnn3JZVTDdVWFepOvg6NGmi2U1S7v2CfLgGVs?type=png)](https://mermaid.live/edit#pako:eNp1UktLw0AQ_ivLnhXvRTyIV2-eNFLWZNoGk2zYbFRsBdOKRPQgIooPlIKIivbgC6WgP2ZpUv-Fs2u0FTGXmZ2d7zHZqVObO0BLhFY8vmzXmJBkZtIKCH6wAvacRXWIJZQjCeH4ghibUM1X1dpUrZZqHarmXq9z3m_vWHSejI5OEAEVD2yJuCIb4LJ0Nzs7zbeuVdLpP9wgtH-5rZrbCP0SLBCapyFgyY2gYUz8vWW2DaFsEJ8HruQC1YqsHApeFRBFRvFj_R6taY_pc--trZKjYbkCYggt_A-BdIMYSiS7xc7OrymTF4sOe_kNFRB6LCiR_lWa73dRLn_aUMl7fvKIihqor-e--8qSRYv_mIDIZh6TaKKXXmQHd8iVpd2CpRb7hsbEctHq8uDfgfzQA82Vn3Ty442PZB9fQBNV3IB5SGSiuwo_BNqeecXBpEZsqEZHCK0K18GVkSIGPPkgfGYKtG4JjbKorIGPtCVMHSZwWn2xprEhC2Y59wdwweNqbXCMQwfnn3JZVTDdVWFepOvg6NGmi2U1S7v2CfLgGVs)

### Reflection と Monitoring を分ける

第2節で作った Reflection は、成果物の質を見て同じ作業をやり直すループだった。Monitoring はその外側で、進行そのものを制御する。

| | Reflection | Monitoring |
|---|---|---|
| 見る対象 | 直近のステップの成果物 | 目的全体に対する進捗 |
| 判断 | このまま採用するか、やり直すか | 続ける / 組み直す / 終える / 人間へ戻す |
| 遷移先 | 同じステップの再実行 | 次のステップ、再計画、完了、エスカレーション |
| 上限 | `max_reflections` | `max_steps`、`max_replans` |

上限を3つとも持たせるのは、それぞれ別の暴走の仕方があるからである。改稿が止まらない、ステップが増え続ける、再計画を繰り返す。どれも「進んでいるように見えて進んでいない」状態になる。

### 5.1 4つの能力を LangGraph へ統合する

`PlannerState` は、目的・成功条件・計画・批評・観測・次の判断を1つの状態として持つ。第3節と第4節では値を返すだけだった LCEL チェーンを、ここでは薄いノード関数から呼び出し、結果を状態更新の辞書へ変換する。

`goal_setting` は `goal_chain`、`plan_task` は `planning_chain` を再利用する。チェーン自体へグラフの責務を持たせず、LangGraphとの接続だけをノード関数が担当する。`initialize_control` は、Reflection の改稿履歴と Monitoring の進捗・判断に使う状態を計画作成後に初期化する。

`execute_step` は LLM を呼ばない。現在のステップを指示文へ組み立て、`messages` へ積むだけの関数である。実際に作業するのは次の `agent` ノードで、第5節では Tool を持たない素の LLM、第6節では HITL と Sandbox を備えた ReAct エージェントを置く。**実行部を差し替えても計画ループ側は変わらない**ようにするための分割である。

`reflect_step` は第2節の Critic と同じ役割だが、判定基準が現在のステップと成功条件になる。`revise` なら `execute_step` へ戻り、指摘を添えて同じステップをやり直す。`max_reflections` に達したら、不十分なまま Monitoring へ渡し、そのステップ単体では直せない問題を再計画で扱う。

`monitor_progress` は、ステップを完了として記録し、進捗カウンタを進め、次の遷移を決める。まず `max_steps` を機械的に確認し、上限に達していれば LLM に聞かずに `escalate` する。上限内であれば、観測結果を成功条件に照らして `MonitorDecision` を返させる。

`human_escalation` は `interrupt()` でグラフを止め、判断に必要な情報を呼び出し側へ返す。第9回の Tool 承認と同じ仕組みだが、問いが違う。Tool 承認が「この操作を実行してよいか」を聞くのに対し、こちらは「この先どう進めるか」を聞く。人間は `continue`、`revise_plan`、`finish` のいずれかを返す。

In [ ]:
# @title 統合グラフの状態とノード
from guarded_agent.state import Context, LongTermMemoryState

NextAction = Literal['continue', 'replan', 'complete', 'escalate']


class PlannerState(LongTermMemoryState, total=False):
    """4つの能力が共有する統合グラフの状態。"""

    # Goal Setting: 何を達成すれば完了か
    user_request: str
    objective: str
    success_criteria: list[str]
    constraints: list[str]

    # Planning: どの順で何をするか
    plan: list[str]
    current_step_index: int

    # Reflection: 直近の成果物で十分か
    critique: str
    revision_notes: list[str]
    reflection_count: int
    max_reflections: int

    # Monitoring: 全体として次に何をすべきか
    completed_steps: list[str]
    observations: list[str]
    last_result: str
    monitor_reason: str
    remaining_gaps: list[str]
    next_action: NextAction
    replan_count: int
    max_replans: int
    max_steps: int

    # 出口
    final_answer: str
    escalated: bool
    human_note: str


def current_step(state: PlannerState) -> str:
    index = state.get('current_step_index', 0)
    plan = state.get('plan', [])
    return plan[index] if index < len(plan) else '追加で実行するステップはない。'


def goal_setting(state: PlannerState) -> dict:
    """Goal Setting の LCEL チェーンをグラフの状態更新へ変換する。"""
    request = state.get('user_request') or str(state['messages'][-1].content)
    result = goal_chain.invoke({'request': request})
    return {
        'user_request': request,
        'objective': result.objective,
        'success_criteria': result.success_criteria,
        'constraints': result.constraints,
    }


def plan_task(state: PlannerState) -> dict:
    """Planning の LCEL チェーンをグラフの状態更新へ変換する。"""
    result = planning_chain.invoke({
        'objective': state['objective'],
        'success_criteria': bullets(state['success_criteria']),
        'constraints': bullets(state['constraints']),
    })
    return {'plan': result.steps, 'current_step_index': 0}


def initialize_control(state: PlannerState) -> dict:
    """Reflection と Monitoring の制御状態を初期化する。"""
    return {
        'completed_steps': [],
        'observations': [],
        'last_result': '',
        'critique': '',
        'revision_notes': [],
        'reflection_count': 0,
        'monitor_reason': '',
        'remaining_gaps': [],
        'next_action': 'continue',
        'replan_count': 0,
        'final_answer': '',
        'escalated': False,
        'human_note': '',
    }


def last_ai_text(state: PlannerState) -> str:
    for message in reversed(state.get('messages', [])):
        if isinstance(message, AIMessage) and message.content:
            return str(message.content)
    return '最終応答を取得できなかった。'


class StepCritique(BaseModel):
    """ステップの成果物に対する批評。"""

    verdict: Literal['accept', 'revise'] = Field(
        description='ステップの目的を満たしていれば accept、やり直すべきなら revise。'
    )
    issues: list[str] = Field(description='不足している点。accept のときは空。')
    guidance: str = Field(description='やり直す場合の指示。1文。')


step_critic = llm.with_structured_output(StepCritique)


def execute_step(state: PlannerState) -> dict:
    """現在のステップを指示文へ組み立て、実行部へ渡す。LLM は呼ばない。"""
    instruction = f"""あなたは大きなタスクの1ステップだけを担当する実行担当です。

Objective:
{state['objective']}

Success Criteria:
{bullets(state['success_criteria'])}

Constraints:
{bullets(state['constraints'])}

Current Step:
{current_step(state)}

これまでの完了ステップ:
{bullets(state.get('completed_steps', []))}
"""
    if state.get('critique'):
        instruction += (
            f'\n前回の成果物への指摘:\n{state["critique"]}\n'
            'この指摘を反映して、同じステップをやり直してください。\n'
        )
    instruction += '\nこのステップの成果物と、残った不確実性を報告してください。'
    return {'messages': [HumanMessage(content=instruction)]}


def reflect_step(state: PlannerState) -> dict:
    """ステップの成果物を批評する。revise なら同じステップをやり直す。"""
    # 長期記憶があれば批評の材料に加える。ユーザーの好みや制約は
    # 成功条件に書かれていないことが多く、批評の側で効いてくる
    memories = state.get('memories', [])
    memory_note = f'\n\nユーザーについて分かっていること:\n{bullets(memories)}' if memories else ''

    result = step_critic.invoke([
        SystemMessage(content=(
            'ステップの成果物を、そのステップの目的と成功条件に照らして批評する。'
            '根拠のない accept は禁止。指摘は1件に絞る。'
        )),
        HumanMessage(content=(
            f'Objective:\n{state["objective"]}\n\n'
            f'Success Criteria:\n{bullets(state["success_criteria"])}\n\n'
            f'Current Step:\n{current_step(state)}\n\n'
            f'過去に受けた指摘:\n{bullets(state.get("revision_notes", []))}'
            f'{memory_note}\n\n'
            f'成果物:\n{state["last_result"]}'
        )),
    ])
    if result.verdict == 'accept':
        return {'critique': ''}
    return {
        'critique': '\n'.join(result.issues + [result.guidance]),
        'revision_notes': state.get('revision_notes', []) + result.issues,
        'reflection_count': state.get('reflection_count', 0) + 1,
    }


def collect_step_result(state: PlannerState) -> dict:
    """実行部の最後の応答を、このステップの成果物として取り出す。"""
    return {'last_result': last_ai_text(state)}


def route_after_reflect(state: PlannerState) -> Literal['execute_step', 'monitor_progress']:
    # 指摘があり、改稿の上限に達していなければ同じステップをやり直す
    if state.get('critique') and state.get('reflection_count', 0) <= state.get('max_reflections', 1):
        return 'execute_step'
    return 'monitor_progress'

In [ ]:
# @title monitor_progress / replan_task / human_escalation / finalize
from langgraph.types import Command, interrupt


class MonitorDecision(BaseModel):
    """観測結果に対する制御判断。"""

    next_action: NextAction = Field(
        description='次のステップへ進む continue、計画を組み直す replan、'
                    '目的を達成した complete、人間へ戻す escalate のいずれか。'
    )
    remaining_gaps: list[str] = Field(description='まだ満たしていない成功条件。')
    reason: str = Field(description='判断理由。観測結果に基づいて簡潔に書く。')


monitor = llm.with_structured_output(MonitorDecision)


def monitor_progress(state: PlannerState) -> dict:
    """ステップを記録し、成功条件と照らして次の遷移を決める。"""
    step_number = state.get('current_step_index', 0) + 1
    completed = state.get('completed_steps', []) + [f'{step_number}. {current_step(state)}']
    observations = state.get('observations', []) + [state['last_result']]

    # ステップを1つ進め、Reflection のカウンタを次のステップ用に戻す
    progress = {
        'completed_steps': completed,
        'observations': observations,
        'current_step_index': step_number,
        'reflection_count': 0,
        'critique': '',
    }

    # 上限判定は LLM に委ねない
    if len(observations) >= state.get('max_steps', 4):
        return {
            **progress,
            'next_action': 'escalate',
            'monitor_reason': '最大ステップ数に到達したため、人間の判断へ戻す。',
            'remaining_gaps': state.get('success_criteria', []),
        }

    decision = monitor.invoke([
        SystemMessage(content=(
            '実行結果を成功条件に照らして監視し、次の制御を選ぶ。\n'
            '- 成功条件を満たしていれば complete\n'
            '- 未達が残っていても、計画の残りステップで解消できるなら continue\n'
            '- 計画をこのまま実行しても成功条件に到達しないと分かる場合だけ replan\n'
            '- 危険な操作、同じ失敗の反復、成功条件を満たしたか判断できない場合は escalate\n'
            '  ただし、依頼の範囲外の未決事項は escalate の理由にしない\n'
            '観測から判断できない条件は、未達ではなく未検証として扱う。'
            '未検証の条件は remaining_gaps に挙げたうえで continue を選ぶ。\n'
            '根拠の弱い complete は禁止する。'
        )),
        HumanMessage(content=(
            f'Objective:\n{state["objective"]}\n\n'
            f'Success Criteria:\n{bullets(state["success_criteria"])}\n\n'
            f'Plan:\n{numbered(state["plan"])}\n\n'
            f'Completed Steps:\n{bullets(completed)}\n\n'
            f'Latest Observation:\n{state["last_result"]}'
        )),
    ])
    return {
        **progress,
        'next_action': decision.next_action,
        'monitor_reason': decision.reason,
        'remaining_gaps': decision.remaining_gaps,
    }


replanning_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        '観測結果と未達条件をもとに、残作業だけの計画へ更新する。'
        'すでに完了した作業は繰り返さない。'
    )),
    ('human', (
        'Objective:\n{objective}\n\n'
        'Success Criteria:\n{success_criteria}\n\n'
        'Completed Steps:\n{completed_steps}\n\n'
        'Remaining Gaps:\n{remaining_gaps}\n\n'
        'Monitoring Reason:\n{monitor_reason}'
    )),
])

replanning_chain = replanning_prompt | llm.with_structured_output(WorkPlan)


def replan_task(state: PlannerState) -> dict:
    """未達のギャップから、残作業だけの計画を作り直す。"""
    if state.get('replan_count', 0) >= state.get('max_replans', 1):
        return {'next_action': 'escalate', 'monitor_reason': '再計画の回数が上限に達した。'}

    result = replanning_chain.invoke({
        'objective': state['objective'],
        'success_criteria': bullets(state['success_criteria']),
        'completed_steps': bullets(state.get('completed_steps', [])),
        'remaining_gaps': bullets(state.get('remaining_gaps', [])),
        'monitor_reason': state.get('monitor_reason', ''),
    })
    return {
        'plan': result.steps,
        'current_step_index': 0,  # 新しい計画の1番目から。完了済みの記録は残る
        'replan_count': state.get('replan_count', 0) + 1,
        'next_action': 'continue',
    }


def human_escalation(state: PlannerState) -> dict:
    """判断を人間へ戻す。Tool 承認とは別の、進め方を問う中断。"""
    response = interrupt({
        'kind': 'planner_escalation',
        'objective': state.get('objective'),
        'success_criteria': state.get('success_criteria', []),
        'completed_steps': state.get('completed_steps', []),
        'remaining_gaps': state.get('remaining_gaps', []),
        'reason': state.get('monitor_reason', '人間の判断が必要'),
        'allowed_decisions': ['continue', 'revise_plan', 'finish'],
    })

    decision = response.get('type')
    if decision == 'finish':
        return {'next_action': 'complete', 'escalated': True,
                'human_note': response.get('message', '人間判断により完了。')}
    if decision == 'revise_plan':
        return {'next_action': 'replan', 'escalated': True,
                'monitor_reason': response.get('message', '人間判断により再計画。'),
                'human_note': response.get('message', '')}
    if decision == 'continue':
        return {'next_action': 'continue', 'escalated': True,
                'human_note': response.get('message', '人間判断により継続。')}
    raise ValueError(f'未対応の人間判断です: {decision}')


def finalize(state: PlannerState) -> dict:
    """完了理由、観測、残リスクをまとめて最終報告にする。"""
    response = llm.invoke([
        SystemMessage(content=(
            'Planner の最終報告を日本語で簡潔に作る。'
            '完了した成果物、根拠、残っているリスクを分けて書く。'
        )),
        HumanMessage(content=(
            f'Objective:\n{state.get("objective")}\n\n'
            f'Completed Steps:\n{bullets(state.get("completed_steps", []))}\n\n'
            f'Observations:\n{bullets(state.get("observations", []))}\n\n'
            f'Remaining Gaps:\n{bullets(state.get("remaining_gaps", []))}\n\n'
            f'人間からの指示:\n{state.get("human_note", "なし")}'
        )),
    ])
    return {'final_answer': str(response.content),
            'messages': [AIMessage(content=str(response.content))]}

In [ ]:
# @title ルーティングとグラフ構築
from guarded_agent.memory import (
    load_memory,
    make_extract_memory,
    validate_memory,
    write_memory,
)
from langgraph.checkpoint.memory import InMemorySaver


def route_after_monitor(
    state: PlannerState,
) -> Literal['execute_step', 'replan_task', 'human_escalation', 'finalize']:
    action = state.get('next_action', 'continue')
    if action == 'complete':
        return 'finalize'
    if action == 'escalate':
        return 'human_escalation'
    if action == 'replan':
        return 'replan_task'
    # 継続でも、計画を使い切っていれば終わる
    if state.get('current_step_index', 0) >= len(state.get('plan', [])):
        return 'finalize'
    return 'execute_step'


def route_after_replan(state: PlannerState) -> Literal['execute_step', 'human_escalation']:
    return 'human_escalation' if state.get('next_action') == 'escalate' else 'execute_step'


def route_after_human(state: PlannerState) -> Literal['execute_step', 'replan_task', 'finalize']:
    action = state.get('next_action', 'continue')
    if action == 'complete':
        return 'finalize'
    if action == 'replan':
        return 'replan_task'
    return 'execute_step'


def build_planner_graph(executor, *, with_memory: bool = False) -> StateGraph:
    """計画ループを組み立てる。executor だけが差し替え可能な実行部。

    `with_memory` は第6節で使う。第5回の長期記憶ノードをループの前後へ足す。
    """
    builder = StateGraph(PlannerState, context_schema=Context)

    builder.add_node('goal_setting', goal_setting)
    builder.add_node('plan_task', plan_task)
    builder.add_node('initialize_control', initialize_control)
    builder.add_node('execute_step', execute_step)
    builder.add_node('agent', executor)
    builder.add_node('collect_step_result', collect_step_result)
    builder.add_node('reflect_step', reflect_step)
    builder.add_node('monitor_progress', monitor_progress)
    builder.add_node('replan_task', replan_task)
    builder.add_node('human_escalation', human_escalation)
    builder.add_node('finalize', finalize)

    if with_memory:
        builder.add_node('load_memory', load_memory)
        builder.add_node('extract_memory', make_extract_memory(llm))
        builder.add_node('validate_memory', validate_memory)
        builder.add_node('write_memory', write_memory)
        builder.add_edge(START, 'load_memory')
        builder.add_edge('load_memory', 'goal_setting')
        builder.add_edge('finalize', 'extract_memory')
        builder.add_edge('extract_memory', 'validate_memory')
        builder.add_edge('validate_memory', 'write_memory')
        builder.add_edge('write_memory', END)
    else:
        builder.add_edge(START, 'goal_setting')
        builder.add_edge('finalize', END)

    builder.add_edge('goal_setting', 'plan_task')
    builder.add_edge('plan_task', 'initialize_control')
    builder.add_edge('initialize_control', 'execute_step')
    builder.add_edge('execute_step', 'agent')
    builder.add_edge('agent', 'collect_step_result')
    builder.add_edge('collect_step_result', 'reflect_step')
    builder.add_conditional_edges(
        'reflect_step',
        route_after_reflect,
        {'execute_step': 'execute_step', 'monitor_progress': 'monitor_progress'},
    )
    builder.add_conditional_edges(
        'monitor_progress',
        route_after_monitor,
        {
            'execute_step': 'execute_step',
            'replan_task': 'replan_task',
            'human_escalation': 'human_escalation',
            'finalize': 'finalize',
        },
    )
    builder.add_conditional_edges(
        'replan_task',
        route_after_replan,
        {'execute_step': 'execute_step', 'human_escalation': 'human_escalation'},
    )
    builder.add_conditional_edges(
        'human_escalation',
        route_after_human,
        {'execute_step': 'execute_step', 'replan_task': 'replan_task', 'finalize': 'finalize'},
    )
    return builder


def simple_executor(state: PlannerState) -> dict:
    """第5節の実行部。Tool を持たない素の LLM。"""
    response = llm.invoke(
        [SystemMessage(content='あなたは指示された1ステップだけを実行する担当です。')]
        + list(state['messages'])
    )
    return {'messages': [response]}


planner_graph = build_planner_graph(simple_executor).compile(checkpointer=InMemorySaver())

display(Image(planner_graph.get_graph().draw_mermaid_png(curve_style=CurveStyle.NATURAL)))

### 5.2 ループを動かす

最終回答だけを見ても、どこで何が起きたかは分からない。`stream(stream_mode='updates')` を使うと、ノードごとの状態更新が順に流れてくるので、遷移をそのまま観察できる。以降の実行はこの `trace()` を通す。

In [ ]:
# @title 遷移を表示するヘルパー
import json
from uuid import uuid4

# 長期記憶の名前空間とモデルを決める実行時コンテキスト。第6節でも使う
planner_context = Context(user_id='session11-user', model=model_id)

# 遷移を追うために表示する状態。値が入った更新だけを出す
TRACE_KEYS = [
    'objective', 'plan', 'critique', 'last_result',
    'next_action', 'monitor_reason', 'remaining_gaps', 'final_answer',
]


def shorten(value, limit: int = 100) -> str:
    text = ' '.join(str(value).split())
    return text if len(text) <= limit else text[:limit] + '…'


def trace(graph, payload, config, context) -> None:
    """ノードの遷移と、主要な状態の変化だけを表示する。"""
    for chunk in graph.stream(payload, config=config, context=context, stream_mode='updates'):
        for node, update in chunk.items():
            if node == '__interrupt__':
                print('  ** 中断: 人間の判断待ち **')
                continue
            print(f'[{node}]')
            if not isinstance(update, dict):
                continue
            for key in TRACE_KEYS:
                if update.get(key):
                    print(f'    {key}: {shorten(update[key])}')

In [ ]:
# @title 実行例: 計画から完了まで
planner_config = {
    'configurable': {'thread_id': f'planner-{uuid4()}'},
    'recursion_limit': 50,  # ループがあるため既定値では足りない
}

trace(
    planner_graph,
    {
        'messages': [HumanMessage(content=WRITING_REQUEST)],
        'user_request': WRITING_REQUEST,
        'max_steps': 4,
        'max_replans': 1,
        'max_reflections': 1,
    },
    planner_config,
    planner_context,
)

snapshot = planner_graph.get_state(planner_config)
if snapshot.interrupts:
    # 上限に達するとここで止まる。再開の仕方は 4.4 で扱う
    print('\n人間の判断待ちで停止:', snapshot.interrupts[0].value['reason'])
else:
    display(Markdown(f'#### 最終報告\n{snapshot.values["final_answer"]}'))

### 5.3 エスカレーションで止める

上の実行は Monitoring が `complete` を選んで終わった。今度は `max_steps` を小さくして、上限に到達させる。`monitor_progress` が LLM に聞かずに `escalate` を返し、`human_escalation` の `interrupt()` でグラフが止まる。

止まった後は、同じ `thread_id` で `Command(resume=...)` を渡して再開する。ここでは `finish` を返して完了させるが、`continue` で続行させることも、`revise_plan` に人間のコメントを添えて計画を組み直させることもできる。

In [ ]:
# @title 上限に到達させて人間へ戻す
escalation_config = {
    'configurable': {'thread_id': f'planner-escalation-{uuid4()}'},
    'recursion_limit': 50,
}

trace(
    planner_graph,
    {
        'messages': [HumanMessage(content=WRITING_REQUEST)],
        'user_request': WRITING_REQUEST,
        'max_steps': 1,  # 1ステップ観測した時点で上限に達する
        'max_replans': 1,
        'max_reflections': 0,
    },
    escalation_config,
    planner_context,
)

pending = planner_graph.get_state(escalation_config).interrupts[0].value
print('\n--- 人間へ渡される情報 ---')
print(json.dumps(pending, ensure_ascii=False, indent=2)[:1200])

In [ ]:
# @title 人間の判断を返して再開する
trace(
    planner_graph,
    Command(resume={'type': 'finish', 'message': '1ステップ分の成果で十分なので完了とする。'}),
    escalation_config,
    planner_context,
)

escalated_state = planner_graph.get_state(escalation_config).values
print('\nescalated:', escalated_state['escalated'])
print('human_note:', escalated_state['human_note'])

### 実装の要点

- **実行部だけを差し替え可能にする**: `build_planner_graph(executor)` は実行部を引数で受け取る。第6節では、ここに HITL と Sandbox 付きの ReAct エージェントを渡すだけで、計画ループのコードは1行も変えない
- **上限判定を LLM に委ねない**: `max_steps` の確認は `monitor_progress` の先頭で機械的に行う。停止の判断まで LLM に任せると、止まるべき場面で止まらない
- **カウンタを進める場所を1か所にする**: `current_step_index` を進めるのは `monitor_progress` だけ。`reflect_step` からのやり直しでは進めないので、同じステップを何度実行しても進捗は誤って進まない
- **再計画は進捗を消さない**: `replan_task` は `current_step_index` を 0 に戻すが、これは新しい計画の1番目という意味で、`completed_steps` と `observations` はそのまま残る。全体の予算は `observations` の件数で見ているため、再計画してもステップ数の上限は巻き戻らない
- **再開は同じ `thread_id` で**: `interrupt()` の中断状態はチェックポインタに保存される。`Command(resume=...)` は同じスレッドへ渡す

### 注意点

- **監視信号が LLM の自己評価だけになりやすい**: `monitor_progress` はモデルの判断に依存している。実際には、テストの成否、静的解析、Tool のエラー、承認の拒否といった機械的な信号を `remaining_gaps` へ反映すべきである
- **再計画が同じ計画を作りうる**: 「完了済みは繰り返さない」と指示しても、モデルは似た計画を返すことがある。`max_replans` はそのための上限で、超えたら人間へ戻す
- **中断中に前提が変わる**: 人間の判断を待つ間も外の状態は動く。待ち時間が長い場合は、再開時に前提を検証し直す

---

## 6. 統合: HITL と Sandbox を計画ループの内側へ

ここまでの実行部は Tool を持たない LLM だった。実際のエージェントはファイルを書き、コマンドを実行し、Web を検索する。目的へ向かって自律的に動くほど、この実行が外部世界へ与える影響も増える。

第9回と第10回で扱った2つの安全境界を、計画ループの内側へ入れる。

- **Human-in-the-Loop**: 副作用を伴う Tool の実行**前**に止め、「実行してよいか」を人間へ問う
- **Sandbox**: 承認された操作でも、**実行中に何ができるか**を隔離環境の制約で縛る

この2つは重なっているようで役割が違う。人間が承認しても、承認されたコマンド自体が想定外の場所へ書き込むことはある。Sandbox はそれを実行環境の側で防ぐ。逆に Sandbox は許可された範囲の操作を止められないので、影響の大きい操作は人間が判断する。

やることは、`build_planner_graph()` に渡す実行部を差し替えるだけである。計画ループのノードもルーティングも変更しない。

In [ ]:
# @title HITL と Sandbox を備えた実行部
# guarded_agent は第9回・第10回で組み立てた実装をパッケージにしたもの。
# make_guarded_agent() は create_agent() の ReAct サブグラフに次の Middleware を載せて返す。
#   - 長期記憶とサンドボックス制約を system prompt へ差し込む dynamic_prompt
#   - shell Tool を提供する ShellToolMiddleware（ExecutionPolicy で隔離の強さを決める）
#   - shell / write_file / file_delete を承認対象にする HumanInTheLoopMiddleware
# ホストで直接動く run_command と python_repl は除外され、コマンド実行の入口は shell だけになる
from guarded_agent.graph import make_guarded_agent
from guarded_agent.hitl import SANDBOX_HITL_INTERRUPT_ON
from guarded_agent.sandbox import make_shell_session_spec
from guarded_agent.tools import DEFAULT_TOOLS, without_host_execution_tools

execution_policy = 'host' # @param ['host', 'docker', 'codex']

shell_spec = make_shell_session_spec(execution_policy)
print('承認が必要な Tool:', [name for name, config in SANDBOX_HITL_INTERRUPT_ON.items() if config])
print('明示的に渡す Tool:', [tool.name for tool in without_host_execution_tools(DEFAULT_TOOLS)])
print('shell Tool は ShellToolMiddleware が追加するため、上の一覧には現れない')
print(f'\n--- shell Tool の制約（{execution_policy}）---\n{shell_spec.note}')

guarded_executor = make_guarded_agent(
    llm,
    tools=DEFAULT_TOOLS,
    execution_policy=execution_policy,
)

In [ ]:
# @title 実行部を差し替えたグラフ
from langchain_openai import OpenAIEmbeddings
from langgraph.store.memory import InMemoryStore

long_term_store = InMemoryStore(
    index={
        'embed': OpenAIEmbeddings(model='text-embedding-3-small'),
        'dims': 1536,
        'fields': ['text'],
    }
)

guarded_planner_graph = build_planner_graph(
    guarded_executor,
    with_memory=True,
).compile(
    checkpointer=InMemorySaver(),
    store=long_term_store,
)

# xray=2 で、agent ノードの中の ReAct サブグラフまで展開する
display(Image(
    guarded_planner_graph.get_graph(xray=2).draw_mermaid_png(curve_style=CurveStyle.NATURAL)
))

### フローの読み方

外側は第5節と同じ計画ループで、`agent` の中身だけが ReAct サブグラフに変わっている。さらに前後へ第5回の長期記憶ノードが付いた。

- `load_memory` がユーザーごとの記憶を読み、`goal_setting` の前に状態へ入れる
- `agent` の中では、モデルがツール呼び出しを出すたびに `HumanInTheLoopMiddleware` が `after_model` で検査する。`shell`、`write_file`、`file_delete` なら `tools` ノードへ進む前に中断する
- 承認されたコマンドは `ShellToolMiddleware` のセッションで実行される。どこで動くかは `ExecutionPolicy` が決める
- `finalize` の後、`extract_memory` が会話から保存候補を取り出し、`validate_memory` が重複と機微情報を落とし、`write_memory` が Store へ書く

この構成では、中断が2種類ある。どちらも `interrupt()` だが、問いも payload も違う。

| | Tool 承認 | エスカレーション |
|---|---|---|
| どこで | `agent` の中（Middleware） | `human_escalation` ノード |
| 問い | このツール実行を許可するか | この先どう進めるか |
| payload の目印 | `action_requests` | `kind: 'planner_escalation'` |
| 返す値 | `{'decisions': [{'type': 'approve'}]}` | `{'type': 'finish'}` など |

再開処理では、payload を見てどちらの中断かを判定する必要がある。

In [ ]:
# @title 実行例: shell の実行前に止まる
SHELL_REQUEST = (
    'shell Tool を使って 1 から 50 までの素数の個数を数え、'
    '使ったコマンドと結果を報告してください。'
)

guarded_config = {
    'configurable': {'thread_id': f'guarded-planner-{uuid4()}'},
    'recursion_limit': 60,
}

trace(
    guarded_planner_graph,
    {
        'messages': [HumanMessage(content=SHELL_REQUEST)],
        'user_request': SHELL_REQUEST,
        'max_steps': 3,
        'max_replans': 1,
        'max_reflections': 1,
    },
    guarded_config,
    planner_context,
)

review_request = guarded_planner_graph.get_state(guarded_config).interrupts[0].value
print('\n--- 承認を求められた内容 ---')
print(json.dumps(review_request, ensure_ascii=False, indent=2)[:1000])

In [ ]:
# @title 承認しながら完了まで進める
def resume_until_done(graph, config, context, *, max_rounds: int = 10):
    """中断のたびに、種類を判定して適切な判断を返す。

    ReAct ループは1ステップで複数回ツールを呼ぶため、承認要求は何度も起きる。
    実際のアプリケーションでは、ここが人間のレビュー画面にあたる。
    """
    for _ in range(max_rounds):
        snapshot = graph.get_state(config)
        if not snapshot.interrupts:
            return snapshot

        value = snapshot.interrupts[0].value
        if isinstance(value, dict) and value.get('kind') == 'planner_escalation':
            print(f'\n>> エスカレーション: {value["reason"]}')
            resume = {'type': 'finish', 'message': '得られた結果で完了とする。'}
        else:
            requests = value['action_requests']
            for request in requests:
                print(f'\n>> 承認: {request["name"]} {request["args"]}')
            resume = {'decisions': [{'type': 'approve'} for _ in requests]}

        trace(graph, Command(resume=resume), config, context)

    raise RuntimeError(f'中断が {max_rounds} 回を超えました')


completed = resume_until_done(guarded_planner_graph, guarded_config, planner_context)
display(Markdown(f'#### 最終報告\n{completed.values["final_answer"]}'))

saved_memories = long_term_store.search(('memories', planner_context.user_id))
print('保存された長期記憶:', [item.value['text'] for item in saved_memories])

### 承認を拒否したらどうなるか

`approve` 以外に `edit`（コマンドを書き換えて実行）と `reject`（実行させず理由を返す）がある。`reject` を返すと、拒否メッセージが ToolMessage としてモデルへ戻り、ReAct ループはそこから続きを考える。

ここで注意したいのは、拒否が計画ループへ自動的には伝わらないことである。モデルは Tool を使わずに答えを作ろうとすることがあり、その成果物を `monitor_progress` が「一応できている」と判定すると、**人間が止めたはずの作業が別経路で完了扱いになる**。拒否は明示的に監視の入力へ流し込む必要がある。第8節の演習で扱う。

In [ ]:
# @title サンドボックスコンテナの後片付け
# DockerExecutionPolicy を選んだときだけ意味がある。
# 中断したまま再開しなかった実行のコンテナは、参照が切れた時点で片付けられる
import gc
import shutil
import subprocess

from guarded_agent.sandbox import SANDBOX_LABEL

gc.collect()

docker_cli = shutil.which('docker')
if execution_policy != 'docker' or docker_cli is None:
    print('後片付けが必要なコンテナはありません')
else:
    leftover = subprocess.run(
        [docker_cli, 'ps', '--quiet', '--filter', f'label={SANDBOX_LABEL}'],
        capture_output=True,
        text=True,
    ).stdout.split()

    if leftover:
        subprocess.run([docker_cli, 'rm', '--force', *leftover], capture_output=True, text=True)

    print('強制削除したサンドボックスコンテナ:', len(leftover))

### 実装の要点

- **計画ループは実行部を知らない**: `build_planner_graph()` に渡すものが変わっただけで、ノードもルーティングも第5節のままである。安全境界を後から足せるのは、実行を1ノードに閉じ込めてあるため
- **状態を共有する**: `PlannerState` は `LongTermMemoryState` を継承しているので、ReAct サブグラフが使う `messages` と `memories` をそのまま共有できる。サブグラフ側で状態を詰め替える必要がない
- **中断の種類を payload で見分ける**: 承認要求には `action_requests`、エスカレーションには `kind` を入れてある。同じ `interrupt()` の仕組みを使う以上、呼び出し側が区別できる目印を payload に持たせる
- **記憶をループの外側に置く**: `load_memory` は `goal_setting` の前、保存は `finalize` の後。目的設定より前に読むことで、過去の制約や好みを成功条件へ反映できる。上の実行で保存された記憶が空なら、それは `validate_memory` が計算結果のような一時的な情報を保存候補から外したためで、想定どおりの動作である

### 注意点

- **レビュー対象と実行対象を一致させる**: 承認画面に出したコマンドと、実際に実行される文字列は同じでなければならない。`edit` で書き換えた場合も、書き換え後の文字列がそのまま実行される
- **迂回路を残さない**: `without_host_execution_tools()` がホスト実行系の Tool を外している。承認が必要な `shell` の隣に、承認なしで同じことができる Tool があると、境界は意味を失う
- **Sandbox の制約をプロンプトへ伝える**: 隔離環境に bash が無い、`/tmp` しか書けない、といった制約はモデルに見えない。`ShellSessionSpec.note` を system prompt へ入れているのはこのため。伝えないと失敗して承認要求を繰り返す
- **`interrupt()` の前に副作用を置かない**: 中断されたノードは再開時に**先頭から再実行**される。`interrupt()` より前に書き込みや外部呼び出しがあると、それが二重に実行される
- **`host` ポリシーは隔離ではない**: 資源上限があるだけで、ファイルシステムもネットワークもホストのままである。学習用途以外では `docker` か `codex` を選ぶ
- **監視は計算の正しさを確かめない**: 上の実行が報告した素数の個数を検算してみるとよい。1から50までの素数は15個だが、モデルの書いたコマンドに誤りがあると別の数が返り、`monitor_progress` はそれを `complete` と判定することがある。監視が見ているのは「観測が成功条件に合っていそうか」であって、計算の再実行ではない

---

## 7. 設計上の注意

### 制御を LLM に委ねすぎない

このノートブックの制御判断は、`reflect_step` と `monitor_progress` の2つがほぼモデルの出力で決まっている。ここには構造的な弱さがある。

- 目的の理解が浅いと、成功条件を満たしていないのに `complete` を選ぶ
- 逆に完璧を求めすぎると、いつまでも `revise` を出して止まらない
- 「約100字」のように LLM が正確に判定できない条件は、永遠に未達と判定される

対策は、機械的に判定できるものは LLM に判定させないことである。ステップ数の上限を先にコードで確認したのはその一例で、同じ考え方をテストの成否、終了コード、スキーマ検証、承認の可否にも広げる。LLM の判断は、機械的に判定できない部分に限って使う。

### 進捗の記録と成果物の質を混ぜない

`monitor_progress` はステップを完了として記録し、同時に次の遷移を決めている。この2つを分けたい場合もある。たとえば「実行はしたが成果は不十分」というステップを完了として記録すると、後から見たときに何が終わったのか分からなくなる。記録には結果の良し悪しも一緒に残す方が、監査でも再開でも扱いやすい。

### 中断は状態設計の一部

`interrupt()` はノードの途中で止まるのではなく、**そのノードを再開時に先頭から再実行する**。副作用を伴う処理を `interrupt()` より前に書くと、承認のたびに二重実行される。中断する可能性のあるノードは、`interrupt()` を先頭付近に置き、それ以降で状態を変える。

### 人間の判断を監視の入力に戻す

承認の拒否、エスカレーションでの指示、`edit` による書き換え。これらはすべて「人間が持っている情報」であり、監視が知らないと同じ提案を繰り返す。`human_note` や `remaining_gaps` へ明示的に書き戻し、次の判断の材料にする。

### コストの見積もり

1ステップあたり、実行（ReAct なので複数回）、批評、監視で最低3回の LLM 呼び出しが発生する。改稿が入ればさらに増える。第5節の実行例では、4ステップで20回前後の呼び出しになった。自律性を上げるほどコストは非線形に増えるので、`max_steps`・`max_reflections`・`max_replans` は品質の調整つまみであると同時に予算の上限でもある。

---

## 8. 小演習

1. **批評を実行結果に置き換える**: 第2節の `critique_node` を、LLM ではなく生成されたコードの `doctest` 実行結果で判定するよう書き換える。テストが通れば `accept`、失敗すればエラー内容を `issues` に入れる。LLM の自己評価と、どちらが早く収束するか比べる
2. **拒否をエスカレーションへ流す**: `resume_until_done` で `{'type': 'reject', 'message': '...'}` を返し、その後の挙動を観察する。モデルが Tool を使わずに答えを作ってしまう場合、`collect_step_result` で ToolMessage の拒否を検出して `next_action` を `escalate` にする経路を足す
3. **コマンドを書き換える**: `{'type': 'edit', 'edited_action': {'name': 'shell', 'args': {'command': ...}}}` で、モデルの提案とは違うコマンドを実行させる。承認画面に出た文字列と実際に実行される文字列が一致していることを確認する
4. **隔離を強くする**: `execution_policy` を `docker` に変えて第6節を再実行する。bash が無い、`/tmp` しか書けない、ネットワークが使えないという制約に対して、エージェントが何回失敗し、何回承認を求めてくるかを数える
5. **記憶を効かせる**: 同じ `user_id` で「回答は必ず箇条書きにしてほしい」と伝える依頼を1回流してから、第6節の依頼を実行する。`load_memory` が読んだ記憶が `reflect_step` の批評をどう変えるか観察する
6. **上限を外してみる**: `max_reflections` を大きくして、批評がどこまで続くかを確かめる。止まらないことを確認したうえで、どんな停止条件なら妥当かを考える

---

## 9. At a Glance

**Reflection**

- **What**: 出力を評価し、その評価をもとに作り直す反復。生成する役割と評価する役割を分離する
- **Why**: 一発生成では、複雑な要求の抜けや細部の誤りが残る
- **Rule of thumb**: 品質・正確さ・制約への適合が、速度とコストより重要なときに使う
- **必須の付属品**: 明示的な評価基準と、`max_iterations` による停止条件

**Goal Setting**

- **What**: 依頼を目的・観測可能な成功条件・制約へ分解する LCEL チェーン
- **Why**: 完了条件がなければ、成果物が十分かを判断できない
- **Rule of thumb**: 成功条件は、判定する側が確かめられる粒度で書く
- **必須の付属品**: 機械判定と LLM 判定の役割分担

**Planning**

- **What**: 目的を実行可能なステップ列へ分解する LCEL チェーン
- **Why**: how が事前に決まっていない依頼では、手順そのものを作る必要がある
- **Rule of thumb**: 手順が既知なら固定ワークフローの方が速く安く安定する
- **必須の付属品**: 計画が外れたと気づく仕組み（Monitoring）

**Monitoring**

- **What**: 4つの能力を LangGraph へ統合し、実行結果から次の制御を選ぶ
- **Why**: 計画は仮説であり、実行中に進捗と未達条件を確かめる必要がある
- **Rule of thumb**: 観測できない条件は未達ではなく未検証として扱う
- **必須の付属品**: 上限（`max_steps` / `max_replans`）と、人間へ戻す経路

**HITL and Sandbox（第9回・第10回の再利用）**

- **What**: 副作用のある Tool の実行前に承認を求め、実行中は隔離環境で縛る
- **Why**: 自律性が上がるほど、想定外の操作が外部へ及ぶ確率が上がる
- **Rule of thumb**: 承認と隔離は計画ループの外側ではなく内側に置く
- **必須の付属品**: 迂回路の排除、レビュー対象と実行対象の一致

---

## 10. Key Takeaways

- Reflection は成果物の質を見る。批評を構造化出力で受け取り、`verdict` を分岐、`issues` を次の入力に使うことで反復を制御できる
- Goal Setting は依頼を目的・成功条件・制約へ変換する。単体では分岐やループがないため、LCEL チェーンで表現できる
- Planning は目的からステップ列を作る。これも単体では LCEL チェーンとし、実行時に LangGraph の状態へ接続する
- Monitoring で4つの能力を LangGraph へ統合する。Reflection と分けることで、止まった理由が「品質不足」か「計画の不一致」かを区別できる
- 記憶があると反復は累積になる。過去の指摘が見えなければ、一度直した問題が戻ってくる
- 停止条件は安全装置ではなく設計要素である。批評は尽きず、再計画は繰り返せるため、上限は外から与える
- 機械的に判定できることを LLM に判定させない。上限の確認、テストの成否、スキーマ検証はコードで行い、LLM の判断はそれ以外に使う
- 実行部を1ノードに閉じ込めておくと、HITL と Sandbox を後から差し込める。安全境界は計画ループの内側の構成要素である
- 中断は2種類ある。Tool 承認は「実行してよいか」、エスカレーションは「どう進めるか」。payload に見分けられる目印を持たせる

---

## 11. Conclusion

今回組み立てたのは、各ステップの成果物を批評し、依頼を目的と成功条件へ変換し、目的までのステップを計画し、進捗を成功条件に照らして次の一手を選ぶループだった。第2節では Session 11 から引き継いだ Reflection を確認し、第3節の Goal Setting と第4節の Planning は単体の LCEL チェーンとして実装した。第5節でその2本のチェーンと Reflection・Monitoring を LangGraph へ統合し、第6節では実行部だけを HITL と Sandbox 付きのエージェントへ差し替えた。

差し替えが1行で済んだのは偶然ではなく、実行を1ノードに閉じ込め、制御を状態として外に出したためである。逆に言えば、目的も進捗も判断も会話文の中にあるエージェントには、後から安全境界を入れる場所が無い。

同時に、このループの制御はまだ LLM の判断に強く依存している。テストの成否や終了コードといった機械的な信号を監視へ組み込むこと、そして上限を明示的に持つことが、実運用へ近づけるうえでの次の一歩になる。

---

## Bibliography

- Training Language Models to Self-Correct via Reinforcement Learning: <https://arxiv.org/abs/2409.12917>
- LangGraph Documentation: <https://www.langchain.com/langgraph>
- LangChain, Human-in-the-loop: <https://docs.langchain.com/oss/python/langchain/hitl>
- LangChain, Middleware: <https://docs.langchain.com/oss/python/langchain/middleware>
- Google DeepResearch (Gemini Feature): <https://gemini.google.com>
- OpenAI, Introducing deep research: <https://openai.com/index/introducing-deep-research/>
- Perplexity, Introducing Perplexity Deep Research: <https://www.perplexity.ai/hub/blog/introducing-perplexity-deep-research>
- SMART Goals Framework: <https://en.wikipedia.org/wiki/SMART_criteria>